## DeepSEM

In [ ]:
!git clone https://github.com/HantaoShu/DeepSEM

Cloning into 'DeepSEM'...
remote: Enumerating objects: 231, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 231 (delta 64), reused 62 (delta 62), pack-reused 160 (from 1)
Receiving objects: 100% (231/231), 31.33 MiB | 15.55 MiB/s, done.
Resolving deltas: 100% (121/121), done.


In [ ]:
%cd /content/DeepSEM

/content/DeepSEM


In [ ]:
!pip install scanpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 65.5 MB/s eta 0:00:00


In [ ]:
!python main.py \
  --task celltype_GRN \
  --data_file demo_data/mESC/data.csv \
  --net_file demo_data/mESC/labels.csv \
  --save_name demo_data/mESC \
  --setting test

dir exist
save dir exist
/content/DeepSEM/src/Model.py:194: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:78.)
  adj_normalized = Tensor(np.eye(adj.shape[0])) - (adj.transpose(0, 1))
epoch: 1 1.0000028908252716 mse_loss: 0.9748145639896393 kl_loss: 0.023504495969973505 sparse_loss: 0.001683778886217624
epoch: 2 0.9625230431556702 mse_loss: 0.9371612668037415 kl_loss: 0.02352098915434908 sparse_loss: 0.0018407446332275867
epoch: 4 0.8737779855728149 mse_loss: 0.8478724360466003 kl_loss: 0.024060172901954502 sparse_loss: 0.001845360908191651
epoch: 5 0.8770170211791992 mse_loss: 0.8510464429855347 kl_loss: 0.024088618403766304 sparse_loss: 0.0018819730030372739
epoch: 7 0.7593521475791931 mse_loss: 0.7321396768093109 kl_loss: 0.02531719405669719 sparse_loss: 0.00189527019392699
epoch: 8 0.74

In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score

# -----------------------------
# Load inferred network
# -----------------------------
pred_df = pd.read_csv("/content/DeepSEM/demo_data/mESC/GRN_inference_result.tsv", sep="\t")

# Ensure correct column names
pred_df = pred_df.rename(columns={
    "TF": "TF",
    "Target": "Target",
    "EdgeWeight": "score"
})

# -----------------------------
# Load ground truth edges
# -----------------------------
gt_df = pd.read_csv("/content/DeepSEM/demo_data/mESC/labels.csv")

gt_df = gt_df.rename(columns={
    "Gene1": "TF",
    "Gene2": "Target"
})

# Convert ground truth to set of tuples
gt_edges = set(zip(gt_df["TF"], gt_df["Target"]))

# -----------------------------
# Create binary labels
# -----------------------------
y_true = [
    1 if (tf, target) in gt_edges else 0
    for tf, target in zip(pred_df["TF"], pred_df["Target"])
]

y_score = pred_df["score"].values

# -----------------------------
# Compute ROC-AUC
# -----------------------------
auc = roc_auc_score(y_true, y_score)

print(f"ROC-AUC score: {auc:.4f}")


ROC-AUC score: 0.5927


In [ ]:
import os
import subprocess
import pandas as pd
from sklearn.metrics import roc_auc_score
from google.colab import files

# Base paths
BASE_DIR = "/content/DeepSEM"
DATA_DIR = f"{BASE_DIR}/demo_data"

# Datasets to process
DATASETS = [
    "dream5-net1",
    "dream5-net2",
    "dream5-net3",
    "dream5-net4",
    "mESC"
]

N_RUNS = 5  # number of repeats per dataset

RESULT_FILE = f"{BASE_DIR}/DeepSEM_AUC_summary.txt"


def compute_auc(pred_file, label_file):
    pred_df = pd.read_csv(pred_file, sep="\t")
    gt_df = pd.read_csv(label_file)

    gt_edges = set(zip(gt_df["Gene1"], gt_df["Gene2"]))

    y_true = [
        1 if (tf, tgt) in gt_edges else 0
        for tf, tgt in zip(pred_df["TF"], pred_df["Target"])
    ]

    y_score = pred_df["EdgeWeight"].values

    return roc_auc_score(y_true, y_score)


with open(RESULT_FILE, "w") as f:
    f.write("Dataset\tRun\tROC_AUC\n")

for dataset in DATASETS:
    print(f"\n🚀 Processing dataset: {dataset}")

    data_path = f"{DATA_DIR}/{dataset}/data.csv"
    label_path = f"{DATA_DIR}/{dataset}/labels.csv"
    save_dir = f"{DATA_DIR}/{dataset}"

    for run in range(1, N_RUNS + 1):
        print(f"  ▶ Run {run}/5")

        # Run DeepSEM
        cmd = [
            "python", "main.py",
            "--task", "celltype_GRN",
            "--data_file", data_path,
            "--net_file", label_path,
            "--save_name", save_dir,
            "--setting", "test"
        ]

        subprocess.run(
            cmd,
            cwd=BASE_DIR,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.STDOUT,
            check=True
        )

        # Compute AUC
        pred_file = f"{save_dir}/GRN_inference_result.tsv"
        auc = compute_auc(pred_file, label_path)

        # Save result immediately (safe against Colab crash)
        with open(RESULT_FILE, "a") as f:
            f.write(f"{dataset}\t{run}\t{auc:.4f}\n")

        print(f"    ✔ AUC = {auc:.4f}")


files.download(RESULT_FILE)



🚀 Processing dataset: mESC
  ▶ Run 1/5
    ✔ AUC = 0.5528
  ▶ Run 2/5
    ✔ AUC = 0.4639
  ▶ Run 3/5
    ✔ AUC = 0.6002
  ▶ Run 4/5
    ✔ AUC = 0.4243
  ▶ Run 5/5
    ✔ AUC = 0.5023


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## DeepRIG

In [2]:
!git clone https://github.com/JChander/DeepRIG

Cloning into 'DeepRIG'...
remote: Enumerating objects: 206, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 206 (delta 63), reused 111 (delta 57), pack-reused 85 (from 1)
Receiving objects: 100% (206/206), 27.35 MiB | 9.00 MiB/s, done.
Resolving deltas: 100% (78/78), done.


In [3]:
%cd /content/DeepRIG

/content/DeepRIG


In [5]:
!python main.py --input_path ./Datasets/mESC/ --output_path ./output/ --cv 5

2026-02-06 03:42:48.775819: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-06 03:42:48.780599: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-06 03:42:48.794673: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770349368.833951    1342 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770349368.844142    1342 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770349368.861494    1342 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [7]:
import pandas as pd
from sklearn.metrics import roc_auc_score

# -----------------------------
# Load inferred network
# -----------------------------
pred_df = pd.read_csv("/content/DeepRIG/output/Inferred_result_mESC.csv")

# Ensure correct column names
pred_df = pred_df.rename(columns={
    "Gene1": "TF",
    "Gene2": "Target",
    "EdgeWeight": "score"
})

# -----------------------------
# Load ground truth edges
# -----------------------------
gt_df = pd.read_csv("/content/DeepRIG/Datasets/mESC/mESC-network.csv")

gt_df = gt_df.rename(columns={
    "Gene1": "TF",
    "Gene2": "Target"
})

# Convert ground truth to set of tuples
gt_edges = set(zip(gt_df["TF"], gt_df["Target"]))

# -----------------------------
# Create binary labels
# -----------------------------
y_true = [
    1 if (tf, target) in gt_edges else 0
    for tf, target in zip(pred_df["TF"], pred_df["Target"])
]

y_score = pred_df["score"].values

# -----------------------------
# Compute ROC-AUC
# -----------------------------
auc = roc_auc_score(y_true, y_score)

print(f"ROC-AUC score: {auc:.4f}")


ROC-AUC score: 0.6076


In [ ]:
import os
import subprocess
import pandas as pd
from sklearn.metrics import roc_auc_score
from google.colab import files

# ==============================
# Paths
# ==============================
BASE_DIR = "/content/DeepRIG"
DATA_DIR = f"{BASE_DIR}/Datasets"
OUTPUT_BASE = f"{BASE_DIR}/output"

DATASETS = [
    "dream5-net1",
    "dream5-net2",
    "dream5-net3",
    "dream5-net4",
    "mESC"
]

N_RUNS = 5

RESULT_FILE = f"{BASE_DIR}/DeepRIG_AUC_summary.txt"


# ==============================
# AUC computation
# ==============================
def compute_auc(pred_file, label_file):
    pred_df = pd.read_csv(pred_file, sep="\t")
    gt_df = pd.read_csv(label_file)

    gt_edges = set(zip(gt_df["Gene1"], gt_df["Gene2"]))

    y_true = [
        1 if (tf, tgt) in gt_edges else 0
        for tf, tgt in zip(pred_df["TF"], pred_df["Target"])
    ]

    y_score = pred_df["EdgeWeight"].values

    return roc_auc_score(y_true, y_score)


# ==============================
# Write header
# ==============================
with open(RESULT_FILE, "w") as f:
    f.write("Dataset\tRun\tROC_AUC\n")


# ==============================
# Main loop
# ==============================
for dataset in DATASETS:
    print(f"\n🚀 Processing dataset: {dataset}")

    input_path = f"{DATA_DIR}/{dataset}"
    label_path = f"{input_path}/labels.csv"

    for run in range(1, N_RUNS + 1):
        print(f"  ▶ Run {run}/{N_RUNS}")

        run_output_dir = f"{OUTPUT_BASE}/{dataset}_run{run}"
        os.makedirs(run_output_dir, exist_ok=True)

        # --------------------------
        # Run DeepRIG
        # --------------------------
        cmd = [
            "python", "main.py",
            "--input_path", input_path,
            "--output_path", run_output_dir,
            "--cv", "5"
        ]

        subprocess.run(
            cmd,
            cwd=BASE_DIR,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.STDOUT,
            check=True
        )

        # --------------------------
        # Prediction file
        # 🔴 CHANGE THIS if needed
        # --------------------------
        pred_file = f"{run_output_dir}/GRN_inference_result.tsv"

        auc = compute_auc(pred_file, label_path)

        # --------------------------
        # Save immediately
        # --------------------------
        with open(RESULT_FILE, "a") as f:
            f.write(f"{dataset}\t{run}\t{auc:.4f}\n")

        print(f"    ✔ AUC = {auc:.4f}")


# ==============================
# Download results
# ==============================
files.download(RESULT_FILE)


## Correlation Based Networks

### a) For mESC

In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.metrics import roc_auc_score
from scipy.stats import pearsonr
from sklearn.utils import resample
import time

In [ ]:
# Load DropSeq expression matrix
expr = pd.read_csv("/content/dropSeq.csv", index_col=0)

# Load ground-truth network
gold = pd.read_csv("/content/mESC-network.csv")

# Remove duplicate edges
gold = gold.drop_duplicates()

# Convert to set of frozensets for fast lookup
gold_edges = set(
    frozenset([g1, g2]) for g1, g2 in zip(gold["Gene1"], gold["Gene2"])
)

In [ ]:
def correlation_auc(expr_matrix, gold_edges):
    genes = expr_matrix.columns.tolist()
    gene_to_idx = {g: i for i, g in enumerate(genes)}

    # Compute correlation matrix (genes × genes)
    corr_matrix = np.corrcoef(expr_matrix.values, rowvar=False)
    corr_matrix = np.abs(corr_matrix)

    y_true = []
    y_score = []

    n_genes = len(genes)

    for i in range(n_genes):
        for j in range(i + 1, n_genes):
            edge = frozenset([genes[i], genes[j]])
            y_score.append(corr_matrix[i, j])
            y_true.append(1 if edge in gold_edges else 0)

    return roc_auc_score(y_true, y_score)

In [ ]:
n_runs = 5
auc_scores = []
time_taken = []

for i in range(n_runs):
    start_time = time.time()

    expr_resampled = resample(expr, replace=True, random_state=i)
    auc = correlation_auc(expr_resampled, gold_edges)

    elapsed_time = time.time() - start_time

    auc_scores.append(auc)
    time_taken.append(elapsed_time)

    print(f"Run {i+1}: AUC = {auc:.4f}, Time = {elapsed_time:.2f} seconds")

Run 1: AUC = 0.4975, Time = 0.29 seconds
Run 2: AUC = 0.4851, Time = 0.27 seconds
Run 3: AUC = 0.5288, Time = 0.28 seconds
Run 4: AUC = 0.4607, Time = 0.26 seconds
Run 5: AUC = 0.4552, Time = 0.26 seconds


### b) For dream5 datasets

In [ ]:
%cd /content

/content


In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations
from scipy.stats import pearsonr
from sklearn.metrics import roc_auc_score
from sklearn.utils import resample

In [ ]:
# Expression data (no row names)
expr = pd.read_csv("dream5-net3-ExpressionData.csv")

# Ground-truth network
gold = pd.read_csv("dream5-net3-network.csv")
gold = gold.drop_duplicates()

In [ ]:
expr_genes = set(expr.columns)

gold_genes = set(gold["Gene1"]).union(set(gold["Gene2"]))

common_genes = sorted(expr_genes.intersection(gold_genes))

print(f"Expression genes : {len(expr_genes)}")
print(f"Gold genes       : {len(gold_genes)}")
print(f"Common genes     : {len(common_genes)}")

# Subset expression matrix
expr = expr[common_genes]

# Convert gold network to edge set
gold_edges = set(
    frozenset([g1, g2]) for g1, g2 in zip(gold["Gene1"], gold["Gene2"])
    if g1 in common_genes and g2 in common_genes
)

Expression genes : 4511
Gold genes       : 1081
Common genes     : 1081


In [ ]:
def correlation_auc(expr_matrix, gold_edges):
    genes = expr_matrix.columns.tolist()

    y_true = []
    y_score = []

    for g1, g2 in combinations(genes, 2):
        corr, _ = pearsonr(expr_matrix[g1], expr_matrix[g2])
        corr = abs(corr)   # DREAM5 uses unsigned confidence

        edge = frozenset([g1, g2])

        y_score.append(corr)
        y_true.append(1 if edge in gold_edges else 0)

    return roc_auc_score(y_true, y_score)

In [ ]:
n_runs = 5
auc_scores = []

for i in range(n_runs):
    expr_boot = resample(expr, replace=True, random_state=i)

    auc = correlation_auc(expr_boot, gold_edges)
    auc_scores.append(auc)

    print(f"Run {i+1}: AUC = {auc:.4f}")

Run 1: AUC = 0.5001
Run 2: AUC = 0.5025
